# California housing capstone — full reproduction
The complete end-to-end analysis from Manifold's California housing capstone:
framing → EDA → cleaning → a regularized linear baseline → three diagnostic-driven upgrades
(spatial features, Tobit, the model zoo) → selection, delivery — and the bonus epilogue
(censored-objective LightGBM + per-prediction SHAP).

Run top to bottom with `estate_train.csv` / `estate_test.csv` in the same folder.
Requires: `pandas numpy scikit-learn scipy xgboost lightgbm shap`.

This notebook follows the exact protocol that produced every number published on the
capstone pages (5-fold CV, seed 42; 20% holdout, seed 42). The lesson pages sometimes show
*compact* code variants for teaching; where they differ, this notebook is the ground truth.

## 1 · Framing: load and look

In [ ]:
import warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", category=FutureWarning)   # sklearn deprecation chatter

train = pd.read_csv("estate_train.csv")
test  = pd.read_csv("estate_test.csv")

print("train:", train.shape, "  test:", test.shape)
print(train.columns.tolist())
print(train["TargetPrice"].describe())
print("rows at cap:", (train.TargetPrice >= 4.9999).sum())   # 808 (4.9%) -> censored!

## 2 · EDA: signal and data-quality landmines

In [ ]:
print(train.corr(numeric_only=True)["TargetPrice"].sort_values(ascending=False).round(3))
# IncomeLevel ~0.69 is the strongest single signal

print("\nnegative populations:", (train.NeighborhoodPop < 0).sum())      # 20 impossible rows
print("missing PropertyAge:", train.PropertyAge.isna().sum())            # 1313 (~8%)
print("AvgOccupancy > 20:", (train.AvgOccupancy > 20).sum(), " max:", round(train.AvgOccupancy.max(), 1))
print("RoomsPerHousehold > 20:", (train.RoomsPerHousehold > 20).sum())
print(train[["TotalRooms","AvgOccupancy","NeighborhoodPop","IncomeLevel"]].skew().round(1))

## 3 · Cleaning (fit on train, apply to test)
Drop the ID, turn impossible/corrupted values into NaN, median-impute with **train** medians.

In [ ]:
def clean(d):
    d = d.drop(columns=["PropertyID"]).copy()
    d.loc[d.NeighborhoodPop < 0, "NeighborhoodPop"] = np.nan      # impossible -> NaN
    d.loc[d.AvgOccupancy > 20, "AvgOccupancy"] = np.nan           # corrupted outliers -> NaN
    d.loc[d.RoomsPerHousehold > 20, "RoomsPerHousehold"] = np.nan
    return d

dc = clean(train)
y = dc.pop("TargetPrice").values
med = dc.median(numeric_only=True)          # train-only statistics
dc = dc.fillna(med)
print("remaining NaN:", int(dc.isna().sum().sum()), "  design matrix:", dc.shape)

## 4 · Baseline & the linear family (one honest CV protocol for everything)

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import r2_score, mean_squared_error

LIN = ['IncomeLevel','PropertyAge','TotalRooms','TotalBedrooms','NeighborhoodPop',
       'AvgOccupancy','Latitude','Longitude','RoomsPerHousehold','BedroomsRatio']

cv = KFold(5, shuffle=True, random_state=42)
def evaluate(model, X):
    pred = cross_val_predict(model, X, y, cv=cv, n_jobs=-1)
    return round(r2_score(y, pred), 3), round(mean_squared_error(y, pred) ** 0.5, 3)

for name, est in [
    ("mean baseline", DummyRegressor()),
    ("linear",        LinearRegression()),
    ("ridge",         RidgeCV(alphas=np.logspace(-3, 3, 40))),
    ("lasso",         LassoCV(n_alphas=60, random_state=42, max_iter=5000)),
    ("elastic net",   ElasticNetCV(l1_ratio=[.2,.5,.8,.95], n_alphas=50, random_state=42, max_iter=5000)),
]:
    print(f"{name:14s}", evaluate(make_pipeline(StandardScaler(), est), dc[LIN]))
# baseline 0.000/1.156 · the whole linear family ties at 0.653/0.681 (n >> p: regularization = insurance)

## 5 · Upgrade 1: spatial feature engineering

In [ ]:
from sklearn.cluster import KMeans

COAST = np.array([(32.7,-117.2),(33.7,-118.2),(34.0,-118.5),(34.4,-119.7),(35.4,-120.9),
                  (36.6,-121.9),(37.8,-122.5),(38.3,-123.0),(39.5,-123.8),(40.8,-124.2),(41.7,-124.2)])

def add_feats(d):
    d = d.copy()
    lat, lon = d.Latitude.values, d.Longitude.values
    d["dist_coast"] = np.min(np.sqrt((lat[:,None]-COAST[:,0])**2 + (lon[:,None]-COAST[:,1])**2), axis=1)
    d["dist_la"] = np.sqrt((lat-34.05)**2 + (lon+118.24)**2)
    d["dist_sf"] = np.sqrt((lat-37.77)**2 + (lon+122.42)**2)
    d["log_pop"] = np.log1p(d.NeighborhoodPop)
    d["log_rooms"] = np.log1p(d.TotalRooms)
    d["inc_per_room"] = d.IncomeLevel / (d.RoomsPerHousehold + 1)
    return d

X_full = add_feats(dc)
print("dist_coast corr with price:", np.corrcoef(X_full.dist_coast, y)[0, 1].round(3))   # -0.47

# the linear model with the spatial features + 12 k-means region intercepts (coords only, no leakage)
region_oh = pd.get_dummies(KMeans(n_clusters=12, n_init=5, random_state=0)
                           .fit_predict(np.c_[dc.Latitude, dc.Longitude]), prefix="reg")
X_sp = pd.concat([dc[LIN], X_full[["dist_coast","dist_sf","dist_la"]],
                  region_oh.set_index(dc.index)], axis=1)
ridge = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 40)))
print("ridge + spatial:", evaluate(ridge, X_sp))   # ~0.67: the +0.02 spatial lift over 0.653

## 6 · The linear ceiling: polynomial expansion

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

poly_ridge = make_pipeline(
    StandardScaler(),
    PolynomialFeatures(degree=2, include_bias=False),
    RidgeCV(alphas=np.logspace(-3, 3, 40)))
print("poly ridge:", evaluate(poly_ridge, X_full[LIN + ["dist_coast","dist_la","dist_sf"]]))   # 0.711 / 0.621

## 7 · Upgrade 2: Tobit (censored regression)
The target is censored at 5.0, so OLS coefficients are attenuated toward zero. Tobit's
likelihood knows "5.0" means "at least 5.0" and recovers the unbiased effects.
(This uses the lesson pages' compact preprocessing — log-transforming the skewed
features — because coefficient comparisons are cleaner on that scale.)

In [ ]:
from scipy.optimize import minimize
from scipy.stats import norm

d_tob = train.drop(columns=["PropertyID"]).copy()
d_tob.loc[d_tob.NeighborhoodPop < 0, "NeighborhoodPop"] = np.nan
d_tob[LIN] = d_tob[LIN].fillna(d_tob[LIN].median())
for c in ["NeighborhoodPop","AvgOccupancy","TotalRooms","RoomsPerHousehold"]:
    d_tob[c] = np.log1p(d_tob[c].clip(lower=0))

U = 5.0
censored = (y >= 4.9999)
X_std = StandardScaler().fit_transform(d_tob[LIN])
Xc = np.c_[np.ones(len(y)), X_std]

ols_beta = np.linalg.lstsq(Xc, y, rcond=None)[0]
x0 = np.r_[ols_beta, np.log(y.std())]

def neg_log_likelihood(params):
    beta, sigma = params[:-1], np.exp(params[-1])
    mu = Xc @ beta
    ll_obs = norm.logpdf(y, mu, sigma)
    ll_cen = norm.logsf((U - mu) / sigma)
    return -np.sum(np.where(censored, ll_cen, ll_obs))

res = minimize(neg_log_likelihood, x0, method="L-BFGS-B")
income_ix = 1 + LIN.index("IncomeLevel")
print("sigma:", np.exp(res.x[-1]).round(3))                                      # 0.674
print("income coef  OLS:", ols_beta[income_ix].round(3),
      "  Tobit:", res.x[income_ix].round(3))                                     # 0.779 -> 0.916 (+18%)

## 8 · Upgrade 3: the model zoo (R² 0.834–0.856) and stacking (0.858)

In [ ]:
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor, StackingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

zoo = {
  "RandomForest":         RandomForestRegressor(n_estimators=300, max_features=0.5,
                              min_samples_leaf=2, n_jobs=-1, random_state=42),
  "HistGradientBoosting": HistGradientBoostingRegressor(max_iter=600, learning_rate=0.05,
                              max_leaf_nodes=31, l2_regularization=1.0, random_state=42),
  "XGBoost":              XGBRegressor(n_estimators=900, learning_rate=0.03, max_depth=6,
                              subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
                              reg_lambda=1.0, n_jobs=-1, random_state=42),
  "LightGBM":             LGBMRegressor(n_estimators=1200, learning_rate=0.03, num_leaves=63,
                              subsample=0.8, colsample_bytree=0.8, min_child_samples=20,
                              reg_lambda=1.0, n_jobs=-1, random_state=42, verbose=-1),
}
for name, model in zoo.items():
    print(f"{name:22s}", evaluate(model, X_full))
# RF 0.834 · HGB 0.846 · XGB 0.856 · LGBM 0.855

stack = StackingRegressor(
    estimators=[("xgb", zoo["XGBoost"]), ("lgbm", zoo["LightGBM"]), ("hgb", zoo["HistGradientBoosting"])],
    final_estimator=RidgeCV(alphas=np.logspace(-2, 2, 20)), cv=cv, n_jobs=-1)
print("Stacking:", evaluate(stack, X_full))   # 0.858 / 0.435 <- best overall

## 9 · Selection checks: cap zone + spatial CV (how honest is 0.858?)

In [ ]:
from sklearn.model_selection import train_test_split, GroupKFold

X_tr, X_ho, y_tr, y_ho = train_test_split(X_full, y, test_size=0.2, random_state=42)
pred = zoo["LightGBM"].fit(X_tr, y_tr).predict(X_ho)
for label, mask in [("normal  ", y_ho < 4.5), ("cap zone", y_ho >= 4.5)]:
    print(label, f"RMSE {np.sqrt(((y_ho[mask]-pred[mask])**2).mean()):.3f}  n={mask.sum()}")
# normal 0.398 (n=3069) vs cap zone 0.911 (n=234): censoring bites, 2.3x worse

regions60 = KMeans(n_clusters=60, n_init=5, random_state=0).fit_predict(np.c_[dc.Latitude, dc.Longitude])
p_spatial = cross_val_predict(zoo["LightGBM"], X_full, y, cv=GroupKFold(5).split(X_full, y, regions60))
print("spatial GroupKFold R2:", round(r2_score(y, p_spatial), 3))   # ~0.70 vs 0.855 random -> leakage gap

## 10 · Bonus: combine the upgrades — a censored objective for LightGBM
Boosting only needs grad + hess of the loss, so the Tobit likelihood drops straight in.
Headline CV *exactly ties* the plain model (the test data is censored too), but cap-zone
RMSE improves 7% and the latent predictions estimate what capped blocks are really worth.

In [ ]:
U, SIGMA = 5.0, 0.4

def tobit_objective(y_true, y_pred):
    cens = y_true >= 4.9999
    g_obs = (y_pred - y_true) / SIGMA**2                 # uncensored: squared-error pull
    h_obs = np.full_like(y_pred, 1 / SIGMA**2)
    z = (y_pred - U) / SIGMA                             # censored: -log P(latent > U)
    lam = np.exp(norm.logpdf(z) - norm.logcdf(z))        # inverse Mills ratio (stable)
    g_cen = -lam / SIGMA
    h_cen = np.clip(lam * (lam + z) / SIGMA**2, 1e-6, None)
    return np.where(cens, g_cen, g_obs), np.where(cens, h_cen, h_obs)

cens_lgbm = LGBMRegressor(objective=tobit_objective, n_estimators=1200, learning_rate=0.03,
                          num_leaves=63, subsample=0.8, colsample_bytree=0.8,
                          min_child_samples=20, reg_lambda=1.0, n_jobs=-1,
                          random_state=42, verbose=-1)

latent_cv = cross_val_predict(cens_lgbm, X_full, y, cv=cv)
print("censored LGBM CV:", round(r2_score(y, np.minimum(latent_cv, U)), 3))   # 0.855 — exact tie

latent = cens_lgbm.fit(X_tr, y_tr).predict(X_ho)
for name, p in [("plain   ", pred), ("censored", np.minimum(latent, U))]:
    m = y_ho >= 4.5
    print(name, f"cap-zone RMSE {np.sqrt(((y_ho[m]-p[m])**2).mean()):.3f}",
          f"bias {(p[m]-y_ho[m]).mean():+.3f}")
# plain 0.911 (bias -0.556) -> censored 0.848 (bias -0.439)

capped = y_ho >= 4.9999
print("latent value of capped blocks: mean", latent[capped].mean().round(3),
      " max", latent[capped].max().round(3))   # ~5.12 / ~7.09 — seeing past the cap

## 11 · Per-prediction SHAP: answer the stakeholder's “why?”

In [ ]:
import shap

lgbm_fit = zoo["LightGBM"].fit(X_tr, y_tr)
explainer = shap.TreeExplainer(lgbm_fit)
sv = explainer.shap_values(X_ho)                                         # (3303, 16)

print("base value:", round(float(explainer.expected_value), 3))          # 2.067
imp = pd.Series(np.abs(sv).mean(0), index=X_full.columns).sort_values(ascending=False)
print(imp.head(8).round(3))                                              # income & dist_coast near-tied on top

pred_ho = lgbm_fit.predict(X_ho)
i = int(np.argmin(np.abs(pred_ho - 4.2)))                                # 'why was I priced 4.2?'
contrib = pd.Series(sv[i], index=X_full.columns).sort_values(key=abs, ascending=False)
print(f"\nblock predicted {pred_ho[i]:.2f}  (base {explainer.expected_value:.2f})")
print(contrib.head(7).round(3))
# shap.plots.waterfall(shap.Explanation(sv[i], explainer.expected_value,
#                      X_ho.iloc[i].values, list(X_full.columns)))  # if running interactively

## 12 · Deliver: predictions on the test set (with the sanity clip)

In [ ]:
test_c = clean(test).fillna(med)                 # TRAIN medians — never re-fit on test
test_f = add_feats(test_c)

final = zoo["LightGBM"].fit(X_full, y)           # refit on ALL training rows
preds = final.predict(test_f)
print("mean:", preds.mean().round(3), " min:", preds.min().round(3), " max:", preds.max().round(3))
print("above cap:", (preds > 5.0).sum())         # ~51 predictions exceed a cap that can't be exceeded

submission = pd.DataFrame({"PropertyID": test["PropertyID"],
                           "TargetPrice": np.clip(preds, 0.15, 5.0)})    # never ship out-of-range
submission.to_csv("predictions.csv", index=False)
print("saved predictions.csv")